# Train WritingPrompts GPT-2 on Colab GPU

This notebook uses the strict-cleaned Action/Drama snapshot:
12,757 train rows, 677 validation rows, and 683 held-out test rows.
Validation and test rows are never added to training.

Upload `writingprompts_training_bundle.zip`. The notebook checks these exact
counts, displays random samples for manual review, and requires explicit audit
confirmation. Each run gets a unique empty output folder and uses 1,024-token
blocks with the college supplemental dataset disabled.

The trainer masks the `Idea:`/`Story:` prefix from loss; training, validation,
and held-out metrics focus on story continuation tokens.

Rows with profanity, sexual violence, direct insults, author notes, and flagged
sexual content near minors are filtered. Remaining genre labels are heuristic.

In [ ]:
# 1. Verify GPU Allocation
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No CUDA GPU attached! In Colab menu: Runtime > Change runtime type > select T4 GPU, then rerun this cell."
    )

gpu_name = torch.cuda.get_device_name(0)
total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ GPU attached: {gpu_name} ({total_mem:.1f} GB VRAM)")


In [ ]:
# 2. Storage Setup (Auto-connect Drive if allowed, or proceed with runtime storage)
from pathlib import Path

use_drive = False
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if Path("/content/drive/MyDrive").is_dir():
        use_drive = True
        print("✅ Google Drive connected. Checkpoints will persist in Drive.")
except Exception as e:
    print(f"ℹ️ Proceeding with local runtime storage (/content).")


In [ ]:
# 3. Upload and extract the WritingPrompts training bundle
from google.colab import files
from pathlib import Path, PurePosixPath
import shutil
import zipfile

upload_root = Path("/content")
bundle_path = upload_root / "writingprompts_training_bundle.zip"
drive_bundle = Path("/content/drive/MyDrive/writingprompts_training_bundle.zip")

if not bundle_path.is_file() and drive_bundle.is_file():
    shutil.copy2(drive_bundle, bundle_path)

if not bundle_path.is_file():
    print("Upload writingprompts_training_bundle.zip from the project folder.")
    uploaded_files = files.upload()
    if bundle_path.name not in uploaded_files:
        raise FileNotFoundError(f"{bundle_path.name} was not uploaded.")

with zipfile.ZipFile(bundle_path) as archive:
    for member in archive.infolist():
        relative_path = PurePosixPath(member.filename.replace(chr(92), "/"))
        if relative_path.is_absolute() or ".." in relative_path.parts:
            raise ValueError(f"Unsafe path in training bundle: {member.filename}")
        destination = upload_root.joinpath(*relative_path.parts)
        if member.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(archive.read(member))

project_dirs = [
    path
    for path in [upload_root, *(item for item in upload_root.iterdir() if item.is_dir() and item.name != "drive")]
    if (path / "Pretrained_GPT-2.py").is_file()
    and (path / "plot_graphs.py").is_file()
    and all((path / "training_ready" / f"{split}.txt").is_file() for split in ("train", "validation", "test"))
    and (path / "training_ready" / "preparation_report.json").is_file()
]
if len(project_dirs) != 1:
    raise FileNotFoundError(
        "Expected exactly one WritingPrompts bundle with trainer, report, and all split files; "
        f"found {len(project_dirs)}. Remove old extracted project folders and rerun this cell."
    )

project_dir = project_dirs[0]
data_dir = project_dir / "training_ready"
train_file = data_dir / "train.txt"
validation_file = data_dir / "validation.txt"
test_file = data_dir / "test.txt"
print(f"Training bundle ready: {project_dir}")
for split, path in (("train", train_file), ("validation", validation_file), ("test", test_file)):
    print(f"{split.title()} rows: {sum(1 for _ in path.open(encoding='utf-8')):,}")

In [ ]:
# 4. Install training dependencies without replacing Colab's CUDA PyTorch
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers>=4.40,<5",
        "accelerate>=0.28,<2",
        "matplotlib>=3.8,<4",
    ],
    check=True,
)
print("Training dependencies installed; the preconfigured Colab PyTorch is unchanged.")

In [ ]:
# 5. Review a deterministic sample before training
from collections import defaultdict
import random
import re

QUALITY_AUDIT_CONFIRMED = True
sample_rng = random.Random(20261015)
for split, path in (("train", train_file), ("validation", validation_file), ("test", test_file)):
    examples_by_genre = defaultdict(list)
    for line in path.read_text(encoding="utf-8").splitlines():
        match = re.search(r"<(action|drama)>", line)
        if match:
            examples_by_genre[match.group(1)].append(line)
    for genre in ("action", "drama"):
        candidates = examples_by_genre[genre]
        if not candidates:
            raise ValueError(f"No {genre} examples found in {split} split.")
        print(f"\\n--- {split.title()} / {genre.title()} ({min(3, len(candidates))} samples) ---")
        for example in sample_rng.sample(candidates, min(3, len(candidates))):
            print(example[:1800], "\\n")

print("Review the samples above. Set QUALITY_AUDIT_CONFIRMED = True and rerun this cell before training.")

In [ ]:
# 6. Train only after reviewing samples and confirming the exact split files
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

if not QUALITY_AUDIT_CONFIRMED:
    raise RuntimeError(
        "Review the displayed samples, set QUALITY_AUDIT_CONFIRMED = True, and rerun the audit cell."
    )

report = json.loads((data_dir / "preparation_report.json").read_text(encoding="utf-8"))
row_counts = {
    split: sum(1 for _ in (data_dir / f"{split}.txt").open(encoding="utf-8"))
    for split in ("train", "validation", "test")
}
expected_counts = {"train": 12757, "validation": 677, "test": 683}
reported_counts = report["written_total"]
if row_counts != expected_counts or row_counts != reported_counts:
    raise ValueError(
        f"Unexpected strict-cleaned WritingPrompts counts: actual={row_counts}, "
        f"reported={reported_counts}, expected={expected_counts}. Do not train this bundle."
    )

train_file = data_dir / "train.txt"
validation_file = data_dir / "validation.txt"
test_file = data_dir / "test.txt"
print(
    f"Verified rows: train={row_counts['train']:,}, "
    f"validation={row_counts['validation']:,}, test={row_counts['test']:,} (held out)"
)

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
if use_drive and Path("/content/drive/MyDrive").is_dir():
    output_dir = Path("/content/drive/MyDrive") / f"storytelling-writingprompts-{run_id}"
else:
    output_dir = Path("/content") / f"storytelling-writingprompts-{run_id}"
if output_dir.exists() and any(output_dir.iterdir()):
    raise FileExistsError(f"Refusing to reuse non-empty output directory: {output_dir}")
output_dir.mkdir(parents=True, exist_ok=True)

os.environ["TRAIN_DATA_FILE"] = str(train_file)
os.environ["ADDITIONAL_TRAIN_DATA_FILE"] = ""
os.environ["EVAL_DATA_FILE"] = str(validation_file)
os.environ["TRAIN_BLOCK_SIZE"] = "1024"
os.environ["TRAIN_OUTPUT_DIR"] = str(output_dir)
os.environ["TRAIN_MODEL_NAME_OR_PATH"] = "gpt2"
os.environ["TRAIN_BATCH_SIZE"] = "1"
os.environ["TRAIN_GRADIENT_ACCUMULATION_STEPS"] = "4"
os.environ["TRAIN_EVAL_BATCH_SIZE"] = "1"
os.environ["TRAIN_SAVE_STEPS"] = "250"
os.environ["TRAIN_MAX_SAMPLES"] = "0"
os.environ["TRAIN_EPOCHS"] = "3"
os.environ["PYTHONUNBUFFERED"] = "1"

print(f"Starting a fresh run in: {output_dir}", flush=True)
process = subprocess.Popen(
    [sys.executable, "-u", str(project_dir / "Pretrained_GPT-2.py")],
    cwd=project_dir,
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
if process.stdout is None:
    raise RuntimeError("Could not read training process output.")
for output_line in process.stdout:
    print(output_line, end="", flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, process.args)

pack_dir = output_dir
print(f"Training complete. Model and tokenizer saved to {pack_dir}")

In [ ]:
# 7. Display training metrics and loss curves
from IPython.display import Image, display

metrics_graph = output_dir / "training_metrics.png"
if metrics_graph.is_file():
    display(Image(filename=str(metrics_graph)))
else:
    print("No training metrics graph was produced.")

In [ ]:
# 8. Evaluate every held-out test row, then inspect generated samples
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
import importlib.util
import json
import math
import random
import re
import sys
import torch

if not test_file.is_file():
    raise FileNotFoundError(f"Held-out test file is missing: {test_file}")

tokenizer = AutoTokenizer.from_pretrained(str(pack_dir))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(str(pack_dir))
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
trainer_path = project_dir / "Pretrained_GPT-2.py"
trainer_spec = importlib.util.spec_from_file_location("storytelling_trainer", trainer_path)
if trainer_spec is None or trainer_spec.loader is None:
    raise ImportError(f"Could not load trainer helpers from {trainer_path}")
trainer_module = importlib.util.module_from_spec(trainer_spec)
sys.modules[trainer_spec.name] = trainer_module
trainer_spec.loader.exec_module(trainer_module)

test_dataset = trainer_module.LineTextDataset(
    tokenizer=tokenizer,
    file_path=str(test_file),
    block_size=1024,
)
if len(test_dataset) != 683:
    raise ValueError(f"Expected 683 held-out test rows, found {len(test_dataset)}")

test_args = TrainingArguments(
    output_dir="/content/writingprompts-test-evaluation",
    do_train=False,
    do_eval=True,
    per_device_eval_batch_size=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
)
test_trainer = Trainer(
    model=model,
    args=test_args,
    data_collator=trainer_module.StoryDataCollator(tokenizer=tokenizer),
    compute_metrics=trainer_module.compute_next_token_accuracy,
    preprocess_logits_for_metrics=trainer_module.preprocess_logits_for_metrics,
)
test_metrics = test_trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test",
)
test_loss = float(test_metrics["test_loss"])
test_metrics["test_perplexity"] = math.exp(test_loss) if test_loss < 709 else float("inf")
(pack_dir / "heldout_test_metrics.json").write_text(
    json.dumps(test_metrics, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Evaluated all {len(test_dataset):,} held-out rows.")
print(f"Test loss: {test_loss:.4f}")
print(f"Test perplexity: {test_metrics['test_perplexity']:.2f}")
print(f"Next-token accuracy: {test_metrics['test_next_token_accuracy']:.4f}")
print("These are token metrics, not proof of story coherence or quality.")

# Compare generated continuations on held-out prompts without exposing target stories.
test_by_genre = {"action": [], "drama": []}
for line in test_file.read_text(encoding="utf-8").splitlines():
    if r"\nStory:" not in line:
        continue
    idea_part = line.split(r"\nStory:", 1)[0]
    genre_match = re.search(r"<(action|drama)>", idea_part)
    if not genre_match or "Idea:" not in idea_part:
        continue
    genre = genre_match.group(1)
    idea = idea_part.split("Idea:", 1)[1].strip()
    prompt = f"<BOS> <{genre}> Idea: {idea}\nStory:"
    if len(tokenizer.encode(prompt)) <= 800:
        test_by_genre[genre].append((idea, prompt))

sample_rng = random.Random(20261015)
for genre, candidates in test_by_genre.items():
    if not candidates:
        raise ValueError(f"No usable held-out {genre} prompts found.")
    for idea, prompt in sample_rng.sample(candidates, min(2, len(candidates))):
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=160,
                min_new_tokens=24,
                do_sample=True,
                temperature=0.65,
                top_p=0.88,
                repetition_penalty=1.12,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
        )
        print(f"\n[{genre}] Idea: {idea}\nGenerated: {generated}\n")

In [ ]:
# 9. Package and download the WritingPrompts fine-tuned model and test metrics
from google.colab import files
from pathlib import Path
import zipfile

if not any((pack_dir / name).is_file() for name in ("model.safetensors", "pytorch_model.bin")):
    raise FileNotFoundError(f"No model weights found in {pack_dir}")

artifact_path = Path("/content/storytelling-writingprompts-model.zip")
model_files = (
    "config.json",
    "generation_config.json",
    "model.safetensors",
    "pytorch_model.bin",
    "added_tokens.json",
    "special_tokens_map.json",
    "tokenizer_config.json",
    "tokenizer.json",
    "vocab.json",
    "merges.txt",
    "eval_results_lm.txt",
    "training_metrics.csv",
    "training_metrics.png",
    "heldout_test_metrics.json",
)
with zipfile.ZipFile(artifact_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in model_files:
        source = pack_dir / name
        if not source.is_file():
            source = Path(output_dir) / name
        if source.is_file():
            archive.write(source, arcname=name)

print(f"Model ZIP ready: {artifact_path.name} ({artifact_path.stat().st_size / (1024 * 1024):.2f} MB)")
print("After review, install these weights as trained model-college and restart the backend.")
files.download(str(artifact_path))